## Spot enrichment analysis

We have extracted cell (nuclear) metrics, spot counts and spot metrics using img-explorer, spot-findr and spot-fitrr. This notebook will take the spot information and look for enrichment of other components on or around a set of selected spots.

In [1]:
# Import necessary packages

import os
import glob
import sys
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import logging

import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
import plotnine as p9
import pandas as pd
from skimage import feature, io
from scipy.stats import norm
from scipy.ndimage import label, find_objects

# Make path for scripts relative to the working directory
sys.path.insert(0, '../src')  # Adjust the path as needed

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
# Define path names and data folders to process

# Define the location of the files to read
parent_directory = "/Users/nestor/Documents/NYU-ISG/microscope-data" # data directory
repo_directory = "/Users/nestor/Documents/NYU-ISG/python_image-analysis" # analysis code directory

# Define the experiment or experiments to analyze
experiment_list = ["05282025_hCEC_gH2AX_IF"]

In [3]:
# User-defined parameters #

# Define the channels where the reference spots are
spot_channel = 2

# Define the channel or channels to measure intensity around the spot
measure_channels = [3, 4]

# Determine also the diameter (in pixels) of the ROI you want to measure around the spot
spot_diameter = 10
spot_radius = spot_diameter // 2  # radius = 5 pixels

In [4]:
# Do some housekeeping first

# Define the folder structure within each experiment
file_folders = {
    "image": "MIPs/MIPs_multichannel/", 
    "mask_n": "segmentation-masks/nuclei_masks/"
}

# Load the function img_loadr() 
from load_files import img_loadr

def extract_base_name(filename):
    """
    Extract the base name from the file name without removing important parts of the name,
    such as the date and experiment information, while stripping the channel prefixes from masks.
    """
    name = os.path.basename(filename)

    # Only remove 'C1-', 'C2-', etc., from mask files
    if name.startswith(("C1-", "C2-", "C3-", "C4-", "C5-")):
        name = name[3:]  # Remove the first 3 characters (e.g., 'C1-', 'C2-', etc)
    
    # Remove the "_cp_masks" suffix for masks or ".ome" from image files if present
    name = name.replace("_cp_masks", "").replace(".ome", "")
    
    # Remove the file extension
    base_name = os.path.splitext(name)[0]
    
    return base_name

In [5]:
# CHUNK 1 # 
# Load maximum intensity projection (MIP) files
# and segmentation masks for images in all experiments listed

# Create a dictionary to store data across multiple experiments
all_experiments_data = {}

# Iterate through each experiment in experiment_list
for experiment in experiment_list:
    # Initialize data structures for the current experiment
    exp_data = {
        "all_images": {"image": [], "mask_n": []},
        "file_names": {"image": [], "mask_n": []},
        "base_names": {"image": [], "mask_n": []},
        "errors": []
    }

    # Read in the experimental groups reference file for each experiment
    exp_groups = pd.read_csv(os.path.join(parent_directory, experiment, "experimental-groups.csv"))

    # Process each file type using img_loadr
    with ThreadPoolExecutor(max_workers=4) as executor:
        for f in file_folders:
            file_path = os.path.join(parent_directory, experiment, file_folders[f])
            files = glob.glob(os.path.join(file_path, "*.tif"))
            results = executor.map(img_loadr, files)

            for result in results:
                image, filename = result
                if image is not None:
                    exp_data["all_images"][f].append(image)
                    exp_data["file_names"][f].append(filename)
                    exp_data["base_names"][f].append(extract_base_name(filename))
                else:
                    exp_data["errors"].append(f"Error loading {filename}")

    # Log the completion of processing for this experiment
    if exp_data["errors"]:
        logging.error(f"{experiment}: Completed with errors in {len(exp_data['errors'])} files.")
    else:
        logging.info(f"{experiment}: All images and masks processed successfully without errors.")
    
    # Store the experiment's data under its name
    all_experiments_data[experiment] = exp_data

# Optional: Summary of processed files for all experiments
for experiment, data in all_experiments_data.items():
    logging.info(f"{experiment}: Processed {len(data['file_names']['image'])} images and {len(data['file_names']['mask_n'])} nuclei masks.")

2025-06-13 10:35:33,959 - INFO - 05282025_hCEC_gH2AX_IF: All images and masks processed successfully without errors.
2025-06-13 10:35:33,961 - INFO - 05282025_hCEC_gH2AX_IF: Processed 150 images and 150 nuclei masks.


In [6]:
# CHUNK 2 #
# Match image indices to mask indices, to make sure they're paired correctly
# as they may not have been read in the same order

# Loop over each experiment in all_experiments_data to match image and mask indices
for experiment, data in all_experiments_data.items():
    logging.info(f"Matching image indices to mask indices for experiment: {experiment}")

    # Access the base names for images and masks in this experiment
    image_base_names = data['base_names']['image']
    mask_base_names = data['base_names']['mask_n']

    # Convert list of mask base names to a dictionary for faster lookup
    mask_name_to_index = {base_name: idx for idx, base_name in enumerate(mask_base_names)}

    # Match each image with its corresponding mask using base names
    matched_indices = []
    missing_masks = 0
    for img_index, img_base_name in enumerate(image_base_names):
        mask_index = mask_name_to_index.get(img_base_name)

        if mask_index is not None:
            matched_indices.append((img_index, mask_index))
        else:
            missing_masks += 1
            logging.warning(f"No matching mask found for image base name: {img_base_name} in experiment {experiment}")

    # Log the results of the matching for this experiment
    logging.info(f"{experiment}: Matched {len(matched_indices)} images with masks. Missing masks for {missing_masks} images.")

    # Optionally, store matched_indices in the experiment's data if needed for downstream analysis
    data["matched_indices"] = matched_indices
    
# Now, matched_indices contains pairs of indices for matched image and mask

2025-06-13 10:35:35,942 - INFO - Matching image indices to mask indices for experiment: 05282025_hCEC_gH2AX_IF
2025-06-13 10:35:35,947 - INFO - 05282025_hCEC_gH2AX_IF: Matched 150 images with masks. Missing masks for 0 images.


In [7]:
# CHUNK 3 #
# Read in spot metrics data generated by spot-filtrr.ipynb

# Initialize a list to hold the spot metrics data
spot_metrics_list = []

# Loop through each experiment in experiment_list
for experiment in experiment_list:
    # Build the path to the experimental-groups.csv file
    metrics_path = os.path.join(repo_directory, "results", experiment, "spot_metrics-postBS.csv")
    
    try:
        # Read the CSV files
        metrics_df = pd.read_csv(metrics_path)
        
        # Append the DataFrame to the list
        spot_metrics_list.append(metrics_df)
        
    except Exception as e:
        logging.error(f"Error reading {metrics_path} for experiment {experiment}: {str(e)}")

# Combine all experimental groups into a single DataFrame
spot_metrics = pd.concat(spot_metrics_list, ignore_index=True)

In [8]:
# Select only "true" spots in the channel indicated by spot_channel
filtr_spot_metrics = spot_metrics[
    (spot_metrics['real_spot'] == "Yes") &
    (spot_metrics['channel'] == spot_channel)
].copy()

In [9]:
# Read in background metrics generated by img-explorer.ipynb
# Initialize a list to hold the spot metrics data
bk_metrics_list = []

# Loop through each experiment in experiment_list
for experiment in experiment_list:
    # Build the path to the experimental-groups.csv file
    metrics_path = os.path.join(repo_directory, "results", experiment, "background_metrics.csv")
    
    try:
        # Read the CSV files
        metrics_df = pd.read_csv(metrics_path)
        
        # Append the DataFrame to the list
        bk_metrics_list.append(metrics_df)
        
    except Exception as e:
        logging.error(f"Error reading {metrics_path} for experiment {experiment}: {str(e)}")

# Combine all experimental groups into a single DataFrame
bk_metrics = pd.concat(bk_metrics_list, ignore_index=True)

In [10]:
# CHUNK 4 # 
# Measure intensity in measure_channel around the true spots, 
# doing background subtraction (background minima) as for the rest of the data

# First, bild a quick lookup: for each (experiment, base_name) → image array
image_lookup = {}
for exp in all_experiments_data:
    base_list = all_experiments_data[exp]["base_names"]["image"]
    img_list  = all_experiments_data[exp]["all_images"]["image"]
    # Pair them up:
    for bname, arr in zip(base_list, img_list):
        image_lookup[(exp, bname)] = arr

# For faster background lookup, index bk_metrics by (experiment, base_name, channel)
bk_index = bk_metrics.set_index(['experiment', 'base_name', 'channel'])['min_intensity']

results = []

# Loop through the filtr_spot_metrics data frame, and 
# for each pair of spot XY coordinates of the reference spot in channel spot_channel
# draw a small circular max (of diameter spot_diameter) 
# and measure the mean and sum intensity on the channel to measure (measure_channel)
for idx, row in filtr_spot_metrics.iterrows():
    exp       = row['experiment']
    bname     = row['base_name']
    x0        = int(round(row['x']))
    y0        = int(round(row['y']))
    
    # Retrieve the multi‐channel NumPy array
    try:
        img = image_lookup[(exp, bname)]
    except KeyError:
        # Fallback if you didn’t preload this image
        # (but ideally, all images are in memory already):
        continue

    # Determine image shape and channel‐axis convention
    if img.ndim == 3 and img.shape[0] in measure_channels + [spot_channel]:
        # channels‐first: (C, H, W)
        channels_first = True
        _, height, width = img.shape
    elif img.ndim == 3 and img.shape[-1] in measure_channels + [spot_channel]:
        # channels‐last: (H, W, C)
        channels_first = False
        height, width, _ = img.shape
    else:
        raise ValueError(f"Unexpected image shape {img.shape} for {exp}/{bname}")
    
    # Define bounding box around the spot (x0, y0)
    y_min = max(0, y0 - spot_radius)
    y_max = min(height, y0 + spot_radius + 1)
    x_min = max(0, x0 - spot_radius)
    x_max = min(width, x0 + spot_radius + 1)

    # Build a small circular mask in that bounding box
    yy, xx = np.ogrid[y_min:y_max, x_min:x_max]
    dist_sq = (xx - x0)**2 + (yy - y0)**2
    circle_mask = dist_sq <= (spot_radius**2)

    # For each requested measure_channel, extract that plane and compute stats
    for mch in measure_channels:
        if channels_first:
            measure_plane = img[mch - 1, :, :]
        else:
            measure_plane = img[..., mch - 1]

        # Crop to the bounding box and apply mask
        patch = measure_plane[y_min:y_max, x_min:x_max]
        vals  = patch[circle_mask]

         # Lookup background minimum for (bname, mch)
        try:
            bg_min = bk_index.loc[(exp, bname, mch)]
        except KeyError:
            bg_min = 0.0  # or raise an error if you prefer

        # Subtract background
        corrected_vals = vals.astype(float) - float(bg_min)
        # (Optionally clip negatives to zero: corrected_vals = np.clip(corrected_vals, 0, None))
        
        if corrected_vals.size > 0:
            mean_int = float(corrected_vals.mean())
            sum_int  = float(corrected_vals.sum())
        else:
            mean_int = np.nan
            sum_int  = np.nan

        results.append({
            'experiment'      : exp,
            'base_name'       : bname,
            'ROI'             : int(row['ROI']),
            'spot_channel'    : spot_channel,
            'measure_channel' : mch,
            'x'               : x0,
            'y'               : y0,
            'bk_min_intensity': float(bg_min),
            'mean_intensity'  : mean_int,
            'sum_intensity'   : sum_int
        })

# Turn results into a Pandas data frame and inspect it
spot_enrichment = pd.DataFrame(results)
print(spot_enrichment.head())

               experiment                                          base_name  \
0  05282025_hCEC_gH2AX_IF  05282025_hCEC_IF_t2_mStayGold-Cen7-ZF1g_1_MMSt...   
1  05282025_hCEC_gH2AX_IF  05282025_hCEC_IF_t2_mStayGold-Cen7-ZF1g_1_MMSt...   
2  05282025_hCEC_gH2AX_IF  05282025_hCEC_IF_t2_mStayGold-Cen7-ZF1g_1_MMSt...   
3  05282025_hCEC_gH2AX_IF  05282025_hCEC_IF_t2_mStayGold-Cen7-ZF1g_1_MMSt...   
4  05282025_hCEC_gH2AX_IF  05282025_hCEC_IF_t2_mStayGold-Cen7-ZF1g_1_MMSt...   

   ROI  spot_channel  measure_channel     x     y  bk_min_intensity  \
0   16             2                3   900  1875             128.0   
1   16             2                4   900  1875               0.0   
2   16             2                3   995  1826             128.0   
3   16             2                4   995  1826               0.0   
4   17             2                3  1330  1820             128.0   

   mean_intensity  sum_intensity  
0     5600.481481       453639.0  
1      171.148148     

In [11]:
# FILE SAVING # 

# Files will be saved in the "results" folder of the analysis repository
results_dir = os.path.join(repo_directory, "results")

# Files for each experiment will be saved in their own folder as .csv files
# They can subsequently be read in and joined as needed
# This is more versatile than combining data from multiple experiments into a single big .csv file

# Get today's date string to use as a suffix if needed
import datetime
date_suffix = datetime.datetime.now().strftime("%m%d%Y")

# Loop through each experiment in experiment_list
for experiment in experiment_list:
    # Define the folder for the current experiment inside the results folder
    experiment_folder = os.path.join(results_dir, experiment)
    
    # Create the folder if it doesn't already exist
    os.makedirs(experiment_folder, exist_ok=True)
    
    # Filter the DataFrames for the current experiment.
    # (Assumes your spot_enrichment DataFrame includes an "experiment" column.)
    spot_enrichment_exp = spot_enrichment[spot_enrichment['experiment'] == experiment]
    
    # Define the default file paths for the CSV files
    spot_enrichment_file = os.path.join(experiment_folder, "spot_enrichment.csv")
    
    # If the spot_enrichment file already exists, add the date suffix to create a new filename.
    if os.path.exists(spot_enrichment_file):
        spot_enrichment_file = os.path.join(experiment_folder, f"spot_enrichment_{date_suffix}.csv")
    
    # Save the DataFrame to its respective CSV files
    spot_enrichment_exp.to_csv(spot_enrichment_file, index=False)